# KITTI — Эксперименты с гиперпараметрами

**RT-DETR-L:** lr=0.01  
**YOLOv8n:** epochs=30 / imgsz=1280 / lr=0.001  

Запускать на Kaggle T4 x1. Время: ~3-4 часа суммарно.

In [ ]:
!git clone https://github.com/TYM4H/cv-kitti-detection.git
%cd cv-kitti-detection
!pip install ultralytics torchmetrics -q

In [ ]:
import torch
print(torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Конвертация датасета (~2 мин)
!python src/dataset/convert_kitti.py \
    --images /kaggle/input/datasets/klemenko/kitti-dataset/data_object_image_2/training/image_2 \
    --labels /kaggle/input/datasets/klemenko/kitti-dataset/data_object_label_2/training/label_2 \
    --out    /kaggle/working/kitti_yolo

## RT-DETR-L — lr=0.01

Проверяем как поведёт себя более высокий фиксированный lr вместо cosine schedule.

In [ ]:
from src.training.train import run

r = run('rtdetr', kitti_yolo_dir='/kaggle/working/kitti_yolo', lr=0.01)
print(r)
# Сохранится в results/logs/rtdetr_e10_sz640_lr0.01.json

## YOLOv8n — epochs=30

Базовая гипотеза: 10 эпох мало, модель не дообучена. Ждём +5..10 mAP50.

In [ ]:
r = run('yolov8n', kitti_yolo_dir='/kaggle/working/kitti_yolo', epochs=30)
print(r)
# Сохранится в results/logs/yolov8n_e30.json

## YOLOv8n — imgsz=1280

Прямой аналог RT-DETR imgsz=1280: сколько даёт повышение разрешения для YOLO.

In [ ]:
# batch уменьшаем до 32 — при imgsz=1280 иначе OOM на T4
r = run('yolov8n', kitti_yolo_dir='/kaggle/working/kitti_yolo', imgsz=1280, batch=32)
print(r)
# Сохранится в results/logs/yolov8n_sz1280.json

## YOLOv8n — lr=0.001

Дефолтный lr0=0.01 в Ultralytics. Понижаем на порядок — ожидаем более плавное обучение, потенциально выше mAP за счёт меньших осцилляций.

In [ ]:
r = run('yolov8n', kitti_yolo_dir='/kaggle/working/kitti_yolo', lr=0.001)
print(r)
# Сохранится в results/logs/yolov8n_lr0.001.json

## Итоговое сравнение всех экспериментов

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob('results/logs/*.json')):
    with open(path) as f:
        d = json.load(f)
    rows.append({
        'name': path.split('/')[-1].replace('.json', ''),
        'model': d.get('model', ''),
        'epochs': d.get('epochs', ''),
        'imgsz': d.get('imgsz', 640),
        'lr': d.get('lr', 'auto'),
        'mAP50': d.get('mAP50', ''),
        'mAP50_95': d.get('mAP50_95', ''),
        'P': d.get('precision', ''),
        'R': d.get('recall', ''),
        'inference_ms': d.get('speed_ms', {}).get('inference', ''),
    })

df = pd.DataFrame(rows).sort_values('mAP50', ascending=False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df.to_string(index=False))

In [ ]:
# График: mAP50 по YOLOv8n экспериментам
import matplotlib.pyplot as plt
import json, glob

yolo_exps = {}
for path in sorted(glob.glob('results/logs/yolov8n*.json')):
    name = path.split('/')[-1].replace('.json', '')
    with open(path) as f:
        d = json.load(f)
    yolo_exps[name] = d.get('mAP50', 0)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(list(yolo_exps.keys()), list(yolo_exps.values()), color='steelblue')
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('mAP50')
ax.set_title('YOLOv8n: влияние гиперпараметров')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('results/plots/yolov8n_experiments.png', dpi=150)
plt.show()

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results_experiments', 'zip', 'results')
print('Сохранено: /kaggle/working/results_experiments.zip')